# Inti, asistente del Consultorio Integra Salud
## Primer avance de proyecto: chatbot basado en reglas con Python y NLTK


Este cuaderno implementa la primera versión de **Inti**, un asistente conversacional para un consultorio privado de medicina general y homeopatía en México. El asistente responde preguntas frecuentes, orienta sobre la consulta de homeopatía, registra citas de prueba en una agenda simulada, consulta citas registradas en la sesión, deriva a atención humana y aplica filtros de seguridad para emergencias, preguntas clínicas y frustración del usuario.

**Aviso:** todos los datos del consultorio (teléfono, correo, dirección, costos y agenda) son ficticios y se usan solo con fines académicos. El asistente no diagnostica ni recomienda tratamientos.

## 1. Objetivo y alcance de la implementación

**Objetivo.** Implementar un chatbot basado en reglas que reciba mensajes de texto, identifique de forma trazable la intención del usuario y devuelva la respuesta asociada a partir de una base de conocimiento estructurada.

**Se incluye en esta entrega**

1. Siete intenciones conversacionales: `saludo`, `consultar_informacion`, `orientacion_homeopatia`, `agendar_cita`, `consultar_cita`, `derivar_humano` y `despedida`.
2. Dos filtros de seguridad que se evalúan antes que cualquier intención: `emergencia` (con una regla secundaria de encuadre clínico) y `contener_frustracion`.
3. Respuesta de contingencia (`fallback`) cuando ninguna regla alcanza el umbral mínimo.
4. Agendamiento simplificado: el usuario indica servicio, día y hora en un solo mensaje; la cita se guarda en memoria durante la sesión.
5. Registro estructurado de cada turno de la conversación.



## 2. Instalación e importación de dependencias

NLTK aporta tres piezas del preprocesamiento: el tokenizador `RegexpTokenizer`, el *stemmer* `SnowballStemmer` para español y la lista de *stopwords* en español. La identificación de intenciones no la hace NLTK: la hacen las reglas definidas en este cuaderno, que trabajan sobre el texto ya procesado.

In [1]:
!pip install -q nltk

In [2]:
import sys
import re
import unicodedata
import pandas as pd
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)

# El proyecto requiere Python 3.10 o superior
assert sys.version_info >= (3, 10), "Se requiere Python 3.10 o superior"
print("Python", sys.version.split()[0], "| NLTK", nltk.__version__)

Python 3.13.16 | NLTK 3.9.1


## 3. Base de conocimiento

La base de conocimiento concentra los datos institucionales del consultorio. Las respuestas se construyen a partir de estos diccionarios, de modo que un cambio de horario o de costo se hace en un solo lugar.

In [3]:
# Datos ficticios del consultorio (solo para fines académicos)
CONSULTORIO = {
    "nombre": "Consultorio Integra Salud",
    "asistente": "Inti",
    "telefono": "55 0000 0000",
    "correo": "contacto@integrasalud.example",
    "direccion": "Av. Ejemplo 123, colonia Centro, Ciudad de México",
}

# Horario de atención: (hora de apertura, hora de cierre); None = cerrado
HORARIO = {
    "lunes": (10, 18), "martes": (10, 18), "miercoles": (10, 18),
    "jueves": (10, 18), "viernes": (10, 18), "sabado": (10, 14),
    "domingo": None,
}
NOMBRE_DIA = {"lunes": "lunes", "martes": "martes", "miercoles": "miércoles",
              "jueves": "jueves", "viernes": "viernes", "sabado": "sábado",
              "domingo": "domingo"}

# Catálogo de servicios con costos de prueba (pesos mexicanos)
SERVICIOS = {
    "medicina_general_primera":     {"nombre": "Consulta de medicina general, primera vez", "modalidad": "medicina_general", "duracion": 45, "costo": 800},
    "medicina_general_subsecuente": {"nombre": "Consulta de medicina general, seguimiento",  "modalidad": "medicina_general", "duracion": 30, "costo": 600},
    "homeopatia_primera":           {"nombre": "Consulta de homeopatía, primera vez",         "modalidad": "homeopatia",       "duracion": 60, "costo": 1000},
    "homeopatia_subsecuente":       {"nombre": "Consulta de homeopatía, seguimiento",          "modalidad": "homeopatia",       "duracion": 45, "costo": 700},
    "certificado_medico":           {"nombre": "Expedición de certificado médico",             "modalidad": "certificado",      "duracion": 30, "costo": 500},
}

METODOS_PAGO = ["efectivo", "tarjeta de débito o crédito en el consultorio", "transferencia electrónica"]

DOCUMENTOS = ["una identificación oficial",
              "la lista de medicamentos que toma actualmente"]
DOCUMENTOS_OPCIONALES = "Si tiene estudios, análisis o recetas previas, llévelos también."

# Agenda simulada: bloques de una hora; se marcan algunos como ocupados
def construir_agenda_base():
    agenda = {}
    for dia, horario in HORARIO.items():
        if horario:
            apertura, cierre = horario
            agenda[dia] = list(range(apertura, cierre))  # el último bloque inicia una hora antes del cierre
    for dia, hora in [("lunes", 10), ("martes", 12), ("jueves", 16), ("sabado", 10)]:
        agenda[dia].remove(hora)  # bloques ocupados de prueba
    return agenda

print(construir_agenda_base())

{'lunes': [11, 12, 13, 14, 15, 16, 17], 'martes': [10, 11, 13, 14, 15, 16, 17], 'miercoles': [10, 11, 12, 13, 14, 15, 16, 17], 'jueves': [10, 11, 12, 13, 14, 15, 17], 'viernes': [10, 11, 12, 13, 14, 15, 16, 17], 'sabado': [11, 12, 13]}


In [4]:
def formatear_lista(elementos):
    # Une una lista en español: "a, b y c"
    if len(elementos) == 1:
        return elementos[0]
    return ", ".join(elementos[:-1]) + " y " + elementos[-1]

def texto_costos(modalidad=None):
    lineas = [f"{s['nombre']}: ${s['costo']:,} MXN ({s['duracion']} minutos)"
              for s in SERVICIOS.values() if modalidad in (None, s["modalidad"])]
    return " ".join(f"{linea}." for linea in lineas)

def texto_horario():
    return ("Lunes a viernes: 10:00 a 18:00 horas. Sábado: 10:00 a 14:00 horas. "
            "Domingo: cerrado.")

DATOS_CONTACTO = (f"teléfono {CONSULTORIO['telefono']} o correo {CONSULTORIO['correo']}, "
                  "de lunes a viernes de 10:00 a 18:00 horas y sábados de 10:00 a 14:00 horas")

# Respuestas de la base de conocimiento, organizadas por clave
RESPUESTAS = {
    "saludo": (
        "Hola, soy Inti, asistente automatizada del Consultorio Integra Salud. "
        "Puedo darle información del consultorio (horarios, ubicación, contacto, servicios, costos, "
        "formas de pago y documentos), orientarle sobre la consulta de homeopatía, registrar una cita "
        "de prueba, consultar una cita registrada o comunicarle con una persona. No puedo dar "
        "orientación médica. Esta es una versión de prueba con datos ficticios: por favor no comparta "
        "datos personales ni de salud. ¿En qué le puedo ayudar?"),
    "info_horarios": f"Nuestro horario es el siguiente. {texto_horario()} ¿Le gustaría agendar una cita?",
    "info_ubicacion": f"Estamos en {CONSULTORIO['direccion']} (dirección ficticia de prueba). ¿Necesita algo más?",
    "info_contacto": f"Puede comunicarse al {DATOS_CONTACTO}. ¿Le ayudo con algo más?",
    "info_servicios": (
        "Ofrecemos consulta de medicina general, consulta de homeopatía (primera vez y seguimiento) "
        "y expedición de certificado médico. Si quiere, le comparto los costos o le explico en qué "
        "consiste la consulta de homeopatía."),
    "info_costos": "Estos son los costos de prueba: {detalle} ¿Desea agendar una cita?",
    "info_metodos_pago": f"Aceptamos {formatear_lista(METODOS_PAGO)}. ¿Le ayudo con algo más?",
    "info_documentos": (f"Para su consulta le recomendamos llevar {formatear_lista(DOCUMENTOS)}. "
                        f"{DOCUMENTOS_OPCIONALES} Le pedimos llegar quince minutos antes para su registro."),
    "info_menu": (
        "Con gusto. Puedo informarle sobre horarios, ubicación, contacto, servicios, costos, "
        "formas de pago o documentos para la consulta. ¿Sobre cuál tema desea saber?"),
    "orientacion_homeopatia": (
        "La consulta de homeopatía es una valoración que realiza el médico del consultorio; la primera "
        "visita dura aproximadamente sesenta minutos e incluye una entrevista detallada sobre sus "
        "antecedentes y hábitos. Es una modalidad complementaria a la medicina general y la decisión "
        "sobre su pertinencia corresponde al profesional. ¿Le gustaría agendar una consulta o conocer los costos?"),
    "agendar_faltan": (
        "Para registrar su cita de prueba necesito {faltantes}. Escríbalo en un solo mensaje, por ejemplo: "
        "\"homeopatía primera vez el jueves a las 11\". Agendamos de lunes a viernes de 10:00 a 17:00 "
        "y sábados de 10:00 a 13:00."),
    "agendar_cerrado": "El consultorio no atiende en {dia}. Puede elegir de lunes a sábado. ¿Qué otro día le acomoda?",
    "agendar_no_disponible": (
        "El {dia} a las {hora} no hay espacio disponible. Los horarios libres ese día son: {libres}. "
        "Escriba de nuevo el servicio, el día y la hora que prefiera."),
    "agendar_exito": (
        "Su cita de prueba quedó registrada. Servicio: {servicio}. Día: {dia}. Hora: {hora}. "
        "Identificador: {id_cita}. {nota}Le pedimos llegar quince minutos antes y llevar una "
        "identificación oficial. Guarde su identificador para consultar la cita. ¿Le ayudo con algo más?"),
    "consultar_pide_id": "Con gusto la reviso. ¿Me comparte el identificador de su cita? Tiene el formato IS101.",
    "consultar_exito": (
        "Encontré su cita {id_cita}. Servicio: {servicio}. Día: {dia}. Hora: {hora}. En esta versión no es "
        "posible cancelar ni reagendar por este medio; para hacerlo comuníquese al consultorio."),
    "consultar_no_encontrada": (
        "No encontré una cita con el identificador {id_cita} en esta sesión. Puedo registrar una cita nueva "
        "o comunicarle con una persona del consultorio. ¿Qué prefiere?"),
    "derivar_humano": (
        "Entiendo, prefiere atención personal. Puede comunicarse al {contacto}. Tenga a la mano su nombre "
        "y, si ya tiene cita, su identificador."),
    "derivar_fuera_alcance": (
        "La cancelación y el reagendamiento de citas todavía no están disponibles en este asistente. "
        "Para hacerlo comuníquese al {contacto}."),
    "emergencia": (
        "Lo que describe puede requerir atención inmediata. Por favor llame ahora al 911 o acuda al servicio "
        "de urgencias más cercano. Si usted o alguien más está en riesgo de hacerse daño, también puede "
        "llamar a la Línea de la Vida al 800 911 2000, disponible las 24 horas. No espere a su próxima cita."),
    "encuadre_clinico": (
        "Como asistente automatizada no puedo valorar síntomas ni recomendar medicamentos o tratamientos; eso "
        "corresponde al médico. Puedo ayudarle a registrar una consulta. Si los síntomas son intensos, empeoran "
        "rápidamente o le preocupan, acuda a urgencias o llame al 911."),
    "contencion_1": (
        "Entiendo que esta situación es incómoda y lamento la molestia. Estoy aquí para ayudarle. ¿Quiere que "
        "lo intentemos de otra forma o prefiere que le comunique con una persona del consultorio?"),
    "contencion_2": (
        "Lamento mucho que la experiencia no haya sido la que esperaba. Para resolverlo mejor, puede comunicarse "
        "con una persona del consultorio al {contacto}."),
    "contencion_3": (
        "Entiendo su molestia y quiero que reciba la mejor atención. Por favor comuníquese al teléfono "
        "{telefono}. Aquí estaré si desea retomar la conversación más tarde."),
    "fallback": (
        "No estoy segura de haber entendido. Puedo ayudarle con información del consultorio, orientación sobre "
        "homeopatía, agendar una cita, consultar una cita registrada o comunicarle con una persona. "
        "¿Qué le gustaría hacer?"),
    "fallback_derivacion": (
        "Sigo sin entender su mensaje y no quiero hacerle perder tiempo. Puede escribirme de otra forma o "
        "comunicarse con una persona del consultorio al {contacto}."),
    "despedida": ("Gracias por escribir al Consultorio Integra Salud. Si necesita algo más, aquí estaré. "
                  "Que tenga un buen día."),
    "entrada_vacia": "No recibí ningún texto. Escriba su pregunta, por ejemplo: \"¿cuál es su horario?\"",
}
print(len(RESPUESTAS), "respuestas cargadas")

28 respuestas cargadas


## 4. Catálogo de intenciones y reglas de identificación

Cada intención conversacional tiene frases de ejemplo. Al preprocesarlas se obtiene un **vocabulario de raíces** por intención, que es la regla de coincidencia del clasificador. Los filtros de seguridad no usan raíces sino **patrones léxicos** (expresiones regulares) sobre el texto normalizado, porque en esos casos importa detectar frases completas como "no puedo respirar".

El orden de evaluación es fijo:

1. Filtro de emergencia y regla de encuadre clínico.
2. Filtro de contención de frustración.
3. Reglas de contexto (identificador de cita, solicitud de cita pendiente, solicitud completa).
4. Clasificación por coincidencia de raíces con umbral mínimo y desempate por prioridad.
5. Fallback orientado.

In [5]:
INTENCIONES = {
    "saludo": [
        "hola", "buenos dias", "buenas tardes", "buenas noches", "que tal",
        "hola inti", "me podria apoyar", "necesito ayuda",
    ],
    "consultar_informacion": [
        "cual es el horario", "horario de atencion", "a que hora abren", "a que hora cierran",
        "trabajan los sabados", "abren el domingo", "cuando atienden",
        "donde estan ubicados", "cual es la direccion", "como llego al consultorio",
        "cual es su telefono", "como los contacto", "tienen correo",
        "que servicios ofrecen", "que especialidades tienen",
        "cuanto cuesta la consulta", "cual es el precio", "costo de la consulta",
        "aceptan tarjeta", "formas de pago", "puedo pagar con transferencia",
        "que documentos debo llevar", "requisitos para la consulta",
    ],
    "orientacion_homeopatia": [
        "que es la homeopatia", "en que consiste la consulta de homeopatia",
        "como es la primera consulta homeopatica", "como funciona la homeopatia",
        "diferencia entre homeopatia y alopatia", "el doctor es homeopata",
    ],
    "agendar_cita": [
        "quiero agendar una cita", "agendar cita", "agendame una cita",
        "me gustaria reservar una consulta", "como pido una cita", "quiero sacar una cita",
        "apartar una cita", "programar una cita",
    ],
    "consultar_cita": [
        "quiero ver mi cita", "consultar mi cita", "cuando es mi cita", "revisa mi cita",
        "ya tengo cita", "datos de mi cita", "a que hora es mi cita", "estado de mi cita",
    ],
    "derivar_humano": [
        "quiero hablar con una persona", "necesito atencion humana", "pasame con recepcion",
        "comunicame con alguien", "prefiero hablar con alguien", "no quiero hablar con un bot",
        "hablar con un asesor", "quiero cancelar mi cita", "necesito reagendar mi cita",
        "cambiar mi cita",
    ],
    "despedida": [
        "gracias", "muchas gracias", "hasta luego", "adios", "nos vemos", "eso es todo", "bye",
    ],
}

# Desempate: si dos intenciones obtienen el mismo puntaje, gana la que aparece primero
PRIORIDAD = ["derivar_humano", "consultar_cita", "agendar_cita", "orientacion_homeopatia",
             "consultar_informacion", "despedida", "saludo"]
UMBRAL = 1  # mínimo de raíces de contenido compartidas para aceptar una intención

# Palabras clave por tema de la intención consultar_informacion
TEMAS = {
    "metodos_pago": ["tarjeta", "efectivo", "transferencia", "pago", "pagar", "credito", "debito", "aceptan"],
    "costos":       ["cuesta", "costo", "precio", "cuanto", "tarifa", "cobran"],
    "documentos":   ["documentos", "llevar", "requisitos", "identificacion", "estudios", "papeles"],
    "horarios":     ["horario", "hora", "abren", "cierran", "abierto", "sabado", "domingo", "atienden", "trabajan"],
    "ubicacion":    ["direccion", "ubicados", "ubicacion", "donde", "llego", "llegar", "encuentran", "mapa"],
    "contacto":     ["telefono", "correo", "contacto", "contacto", "contactar", "llamar", "numero", "whatsapp", "email"],
    "servicios":    ["servicios", "ofrecen", "especialidades", "especialidad"],
}

In [6]:
# Filtros de seguridad: patrones sobre el texto normalizado (sin acentos, minúsculas)
PATRONES_EMERGENCIA = [
    r"\b(dolor|duele|presion|opresion)\b.{0,30}\bpecho\b",
    r"\bno puedo respirar\b", r"\bme falta el aire\b", r"\bme ahogo\b", r"\bdificultad para respirar\b",
    r"\bsangrando\b", r"\bhemorragia\b",
    r"\bdesmay\w*", r"\bperdi el conocimiento\b", r"\binconsciente\b", r"\bconvulsi\w*",
    r"\binfarto\b", r"\bderrame cerebral\b", r"\bembolia\b", r"\bparalisis\b",
    r"\bme siento muy mal\b", r"\bsiento que me muero\b", r"\bme voy a morir\b", r"\bme estoy muriendo\b",
    r"\bhacerme dano\b", r"\blastimarme\b", r"\bsuicid\w*", r"\bquitarme la vida\b",
    r"\bno quiero vivir\b", r"\bmatarme\b",
    r"\bsobredosis\b", r"\benvenen\w*", r"\bintoxica\w*",
    r"\bemergencia\b", r"\burgencias?\b", r"\burgente\b",
]

# Regla secundaria: síntomas o solicitudes clínicas no urgentes
PATRONES_ENCUADRE = [
    r"\b(fiebre|tos|gripa|gripe|diarrea|vomit\w*|nausea\w*|mareo\w*|sintoma\w*|infeccion|alergia|comezon|dolor|duele\w*|inflamad\w*)\b",
    r"\bme siento mal\b",
    r"\b(que|cual) (medicamento|medicina|remedio|pastilla|tratamiento)\b",
    r"\bque (me|le|puedo|debo) (tomo|tomar|doy|dar)\b",
    r"\bdosis\b", r"\bdiagnostic\w*", r"\bme receta\w*", r"\bes grave\b",
]

# Diccionario reducido para el filtro de contención, organizado por familia
LEXICO_CONTENCION = {
    "lenguaje_agresivo": [
        "idiota", "estupido", "estupida", "imbecil", "tarado", "tarada", "menso", "mensa",
        "pendejo", "pendeja", "maldito", "maldita", "inutil", "porqueria", "basura",
    ],
    "enojo_frustracion": [
        "estoy harto", "estoy harta", "me tienes harto", "me tienes harta", "mala atencion",
        "pesimo servicio", "pesima atencion", "no sirve", "no sirves", "pierdo mi tiempo",
        "perdiendo mi tiempo", "no entiendes nada", "que fastidio", "que coraje",
        "estoy molesto", "estoy molesta", "estoy enojado", "estoy enojada",
    ],
    "impaciencia_usabilidad": [
        "llevo mucho esperando", "nadie responde", "nadie contesta", "siempre lo mismo",
        "otra vez el bot", "ya me canse", "tercera vez",
    ],
}

# Regla de alcance: cancelación o reagendamiento se derivan a una persona
PATRON_FUERA_ALCANCE = r"\b(cancel\w*|reagend\w*|cambiar mi cita|mover mi cita)\b"

## 5. Funciones de preprocesamiento con NLTK

El preprocesamiento sigue el pipeline visto en clase: normalizar, tokenizar, eliminar *stopwords* y reducir a raíces.

Se hacen dos ajustes a la lista de *stopwords* de NLTK, justificados por las pruebas del cuaderno base:

1. Se **conservan** "cuando", "donde" y "todo", porque distinguen intenciones ("¿cuándo es mi cita?", "¿dónde están?", "eso es todo").
2. Se **agregan** verbos de cortesía muy frecuentes ("quiero", "necesito", "quisiera") y el nombre del asistente, que en el cuaderno base provocaban coincidencias sin contenido.

In [7]:
tokenizer = RegexpTokenizer(r"\w+")   # separa en palabras alfanuméricas
stemmer = SnowballStemmer("spanish")   # reduce cada palabra a su raíz

def quitar_acentos(texto):
    texto = unicodedata.normalize("NFD", texto)
    return texto.encode("ascii", "ignore").decode("utf-8")

def normalizar_texto(texto):
    # Minúsculas, sin acentos, sin signos de puntuación y con espacios simples
    texto = quitar_acentos(texto.lower())
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()

CONSERVAR = {"cuando", "donde", "todo"}
AGREGAR = {"quiero", "quisiera", "gustaria", "necesito", "favor", "podria", "puedo", "puede",
           "pueden", "saber", "usted", "ustedes", "inti", "consultorio", "debo"}
STOPWORDS = ({normalizar_texto(p) for p in stopwords.words("spanish")} - CONSERVAR) | AGREGAR

def preprocesar(texto):
    # Devuelve los tokens, las palabras de contenido y sus raíces
    tokens = tokenizer.tokenize(normalizar_texto(texto))
    contenido = [t for t in tokens if t not in STOPWORDS]
    raices = [stemmer.stem(t) for t in contenido]
    return {"tokens": tokens, "contenido": contenido, "raices": raices}

ejemplo = preprocesar("¡Hola! ¿Cuándo es mi cita de Homeopatía?")
for clave, valor in ejemplo.items():
    print(f"{clave:10s} {valor}")

tokens     ['hola', 'cuando', 'es', 'mi', 'cita', 'de', 'homeopatia']
contenido  ['hola', 'cuando', 'cita', 'homeopatia']
raices     ['hol', 'cuand', 'cit', 'homeopati']


In [8]:
# Vocabulario de raíces por intención: se construye a partir de los ejemplos del catálogo
def construir_vocabulario(intenciones):
    return {nombre: {r for frase in frases for r in preprocesar(frase)["raices"]}
            for nombre, frases in intenciones.items()}

VOCABULARIO = construir_vocabulario(INTENCIONES)
# Las palabras clave de los temas también identifican la intención consultar_informacion
RAICES_TEMA = {tema: {stemmer.stem(normalizar_texto(p)) for p in palabras} for tema, palabras in TEMAS.items()}
VOCABULARIO["consultar_informacion"] |= set().union(*RAICES_TEMA.values())

for intencion in PRIORIDAD:
    print(f"{intencion:23s} {sorted(VOCABULARIO[intencion])}")

derivar_humano          ['algui', 'asesor', 'atencion', 'bot', 'cambi', 'cancel', 'cit', 'comunicam', 'habl', 'human', 'pasam', 'person', 'prefier', 'reagend', 'recepcion']
consultar_cita          ['cit', 'consult', 'cuand', 'dat', 'hor', 'revis', 'ver']
agendar_cita            ['agend', 'agendam', 'apart', 'cit', 'consult', 'pid', 'program', 'reserv', 'sac']
orientacion_homeopatia  ['alopati', 'cons', 'consult', 'diferent', 'doctor', 'funcion', 'homeopat', 'homeopati', 'primer']
consultar_informacion   ['abiert', 'abren', 'acept', 'atencion', 'atiend', 'cierr', 'cobr', 'consult', 'contact', 'corre', 'cost', 'credit', 'cuand', 'cuant', 'cuest', 'debit', 'direccion', 'document', 'doming', 'dond', 'efect', 'email', 'encuentr', 'especial', 'estudi', 'form', 'hor', 'horari', 'identif', 'llam', 'lleg', 'llev', 'map', 'numer', 'ofrec', 'pag', 'papel', 'preci', 'requisit', 'sab', 'servici', 'tarif', 'tarjet', 'telefon', 'trabaj', 'transferent', 'ubic', 'whatsapp']
despedida               ['ad

## 6. Extracción de entidades

Las entidades se obtienen con diccionarios y expresiones regulares:

| Entidad | Ejemplo | Técnica |
|---|---|---|
| `tema` | horarios, costos, documentos | Raíces de palabras clave por tema |
| `servicio` | homeopatia_primera | Expresiones regulares con sinónimos |
| `dia` | jueves | Lista cerrada de días |
| `hora` | 11:00 | Expresión regular ("a las 11", "4 pm", "16:00") |
| `id_cita` | IS101 | Expresión regular |

In [9]:
DIAS = ["lunes", "martes", "miercoles", "jueves", "viernes", "sabado", "domingo"]
PATRON_HORA = re.compile(
    r"\b(?:a las|a la|las)\s+(\d{1,2})(?::(\d{2}))?\b"   # a las 11, a las 4:00
    r"|\b(\d{1,2}):(\d{2})\b"                             # 16:00
    r"|\b(\d{1,2})\s*(am|pm|a m|p m|hrs|horas|h)\b"      # 4 pm, 11 hrs
)
PATRON_ID = re.compile(r"\bis\s?(\d{3})\b")

def texto_para_entidades(texto):
    # Igual que normalizar_texto, pero conserva los dos puntos de las horas
    texto = quitar_acentos(texto.lower())
    texto = re.sub(r"[^a-z0-9:\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()

def extraer_hora(texto):
    m = PATRON_HORA.search(texto)
    if not m:
        return None
    hora = int(m.group(1) or m.group(3) or m.group(5))
    minutos = int(m.group(2) or m.group(4) or 0)
    sufijo = (m.group(6) or "").replace(" ", "")
    if (sufijo == "pm" or "de la tarde" in texto) and hora < 12:
        hora += 12
    elif sufijo != "am" and 1 <= hora <= 7:
        hora += 12  # en el horario del consultorio, de 1 a 7 se interpreta como tarde
    if hora > 23 or minutos > 59:
        return None
    return (hora, minutos)

def extraer_servicio(texto):
    # Devuelve (clave del servicio, si el tipo se asumió por omisión)
    seguimiento = re.search(r"\b(seguimiento|subsecuente|control|segunda vez|segunda consulta)\b", texto)
    if re.search(r"\bcertificado", texto):
        return "certificado_medico", False
    if re.search(r"\bhomeopat", texto):
        modalidad = "homeopatia"
    elif re.search(r"\b(medicina general|medico general|consulta general|medicina)\b", texto):
        modalidad = "medicina_general"
    else:
        return None, False
    if seguimiento:
        return f"{modalidad}_subsecuente", False
    explicito = re.search(r"\bprimera\b", texto) is not None
    return f"{modalidad}_primera", not explicito

def extraer_tema(raices):
    puntajes = {tema: len(set(raices) & r) for tema, r in RAICES_TEMA.items()}
    mejor = max(puntajes, key=puntajes.get)  # empate: gana el primero del diccionario
    return mejor if puntajes[mejor] > 0 else None

def extraer_entidades(texto, raices):
    t = texto_para_entidades(texto)
    servicio, tipo_asumido = extraer_servicio(t)
    dia = next((d for d in DIAS if re.search(rf"\b{d}\b", t)), None)
    id_match = PATRON_ID.search(t)
    entidades = {
        "tema": extraer_tema(raices),
        "servicio": servicio,
        "tipo_asumido": tipo_asumido,
        "dia": dia,
        "hora": extraer_hora(t),
        "id_cita": f"IS{id_match.group(1)}" if id_match else None,
    }
    return {k: v for k, v in entidades.items() if v not in (None, False)}

for frase in ["Quiero homeopatía el sábado a las 11", "¿cuánto cuesta el certificado?",
              "mi cita es la IS101", "medicina general de seguimiento el viernes 4 pm"]:
    print(f"{frase:50s} -> {extraer_entidades(frase, preprocesar(frase)['raices'])}")

Quiero homeopatía el sábado a las 11               -> {'tema': 'horarios', 'servicio': 'homeopatia_primera', 'tipo_asumido': True, 'dia': 'sabado', 'hora': (11, 0)}
¿cuánto cuesta el certificado?                     -> {'tema': 'costos', 'servicio': 'certificado_medico'}
mi cita es la IS101                                -> {'id_cita': 'IS101'}
medicina general de seguimiento el viernes 4 pm    -> {'servicio': 'medicina_general_subsecuente', 'dia': 'viernes', 'hora': (16, 0)}


## 7. Lógica de identificación de intenciones

La función `identificar_intencion` aplica las capas en el orden definido y devuelve la intención junto con la **regla** que la produjo. Guardar la regla permite auditar cada decisión del chatbot.

In [10]:
def coincide(patrones, texto):
    return any(re.search(p, texto) for p in patrones)

def detectar_contencion(texto):
    for familia, terminos in LEXICO_CONTENCION.items():
        if any(re.search(rf"\b{re.escape(t)}\b", texto) for t in terminos):
            return familia
    return None

def clasificar_por_raices(raices):
    conjunto = set(raices)
    puntajes = {i: len(conjunto & VOCABULARIO[i]) for i in PRIORIDAD}
    mejor = max(PRIORIDAD, key=lambda i: puntajes[i])  # max conserva el primero en caso de empate
    if puntajes[mejor] < UMBRAL:
        return "fallback", puntajes
    return mejor, puntajes

def identificar_intencion(texto, sesion):
    norm = normalizar_texto(texto)
    if not norm:
        return {"intencion": "entrada_vacia", "regla": "validacion_entrada", "entidades": {}, "puntajes": {}}

    # Capa 1: filtro de emergencia y encuadre clínico
    if coincide(PATRONES_EMERGENCIA, norm):
        return {"intencion": "emergencia", "regla": "filtro_emergencia", "entidades": {}, "puntajes": {}}
    if coincide(PATRONES_ENCUADRE, norm):
        return {"intencion": "encuadre_clinico", "regla": "filtro_emergencia_encuadre", "entidades": {}, "puntajes": {}}

    # Capa 2: filtro de contención
    familia = detectar_contencion(norm)
    if familia:
        return {"intencion": "contener_frustracion", "regla": f"filtro_contencion:{familia}",
                "entidades": {}, "puntajes": {}}

    pre = preprocesar(texto)
    ent = extraer_entidades(texto, pre["raices"])

    # Capa 3: reglas de contexto
    if "id_cita" in ent:
        return {"intencion": "consultar_cita", "regla": "contexto_id_cita", "entidades": ent, "puntajes": {}}
    datos_cita = any(k in ent for k in ("servicio", "dia", "hora"))
    if sesion["pendiente"] == "agendar_cita" and datos_cita:
        return {"intencion": "agendar_cita", "regla": "contexto_pendiente", "entidades": ent, "puntajes": {}}
    if all(k in ent for k in ("servicio", "dia", "hora")):
        return {"intencion": "agendar_cita", "regla": "entidades_completas", "entidades": ent, "puntajes": {}}

    # Capa 4: clasificación por raíces con umbral
    intencion, puntajes = clasificar_por_raices(pre["raices"])
    regla = "raices" if intencion != "fallback" else "umbral_no_alcanzado"
    return {"intencion": intencion, "regla": regla, "entidades": ent, "puntajes": puntajes}

print(identificar_intencion("¿A qué hora abren?", {"pendiente": None}))

{'intencion': 'consultar_informacion', 'regla': 'raices', 'entidades': {'tema': 'horarios'}, 'puntajes': {'derivar_humano': 0, 'consultar_cita': 1, 'agendar_cita': 0, 'orientacion_homeopatia': 0, 'consultar_informacion': 2, 'despedida': 0, 'saludo': 0}}


## 8. Gestión de la sesión y función principal del chatbot

La sesión guarda el estado mínimo de la conversación: la agenda simulada, las citas registradas, el contador de frustración, los fallbacks consecutivos, una solicitud pendiente y el historial. La función `responder` recibe un mensaje, identifica la intención, ejecuta la acción asociada y registra el turno.

In [11]:
def crear_sesion():
    return {
        "agenda": construir_agenda_base(),
        "citas": {},
        "siguiente_id": 101,
        "contador_frustracion": 0,
        "fallbacks_consecutivos": 0,
        "pendiente": None,
        "historial": [],
    }

def formato_hora(hora):
    return f"{hora[0]:02d}:{hora[1]:02d}"

def gestionar_agendamiento(ent, sesion):
    faltantes = [nombre for clave, nombre in [("servicio", "el servicio"), ("dia", "el día"), ("hora", "la hora")]
                 if clave not in ent]
    if faltantes:
        sesion["pendiente"] = "agendar_cita"
        return RESPUESTAS["agendar_faltan"].format(faltantes=formatear_lista(faltantes))

    dia, hora = ent["dia"], ent["hora"]
    if dia not in sesion["agenda"]:
        sesion["pendiente"] = "agendar_cita"
        return RESPUESTAS["agendar_cerrado"].format(dia=NOMBRE_DIA[dia])

    libres = sesion["agenda"][dia]
    if hora[1] != 0 or hora[0] not in libres:
        sesion["pendiente"] = "agendar_cita"
        texto_libres = formatear_lista([f"{h:02d}:00" for h in libres]) if libres else "ninguno"
        return RESPUESTAS["agendar_no_disponible"].format(dia=NOMBRE_DIA[dia], hora=formato_hora(hora),
                                                          libres=texto_libres)

    # Registro de la cita: se ocupa el bloque y se genera el identificador
    id_cita = f"IS{sesion['siguiente_id']}"
    sesion["siguiente_id"] += 1
    libres.remove(hora[0])
    servicio = SERVICIOS[ent["servicio"]]["nombre"]
    sesion["citas"][id_cita] = {"servicio": servicio, "dia": NOMBRE_DIA[dia], "hora": formato_hora(hora)}
    sesion["pendiente"] = None
    sesion["contador_frustracion"] = 0  # un flujo completado reinicia el contador
    nota = ("La registré como primera vez; si es de seguimiento, indíquelo al llegar. "
            if ent.get("tipo_asumido") else "")
    return RESPUESTAS["agendar_exito"].format(servicio=servicio, dia=NOMBRE_DIA[dia], hora=formato_hora(hora),
                                              id_cita=id_cita, nota=nota)

def gestionar_consulta_cita(ent, sesion):
    if "id_cita" not in ent:
        sesion["pendiente"] = "consultar_cita"
        return RESPUESTAS["consultar_pide_id"]
    sesion["pendiente"] = None
    cita = sesion["citas"].get(ent["id_cita"])
    if cita is None:
        return RESPUESTAS["consultar_no_encontrada"].format(id_cita=ent["id_cita"])
    return RESPUESTAS["consultar_exito"].format(id_cita=ent["id_cita"], **cita)

def responder_informacion(ent):
    tema = ent.get("tema")
    if tema is None:
        return RESPUESTAS["info_menu"]
    if tema == "costos":
        modalidad = SERVICIOS[ent["servicio"]]["modalidad"] if "servicio" in ent else None
        return RESPUESTAS["info_costos"].format(detalle=texto_costos(modalidad))
    return RESPUESTAS[f"info_{tema}"]

def responder_contencion(sesion):
    sesion["contador_frustracion"] += 1
    nivel = min(sesion["contador_frustracion"], 3)
    return RESPUESTAS[f"contencion_{nivel}"].format(contacto=DATOS_CONTACTO, telefono=CONSULTORIO["telefono"])

In [12]:
def responder(mensaje, sesion):
    texto = mensaje if isinstance(mensaje, str) else ""
    resultado = identificar_intencion(texto, sesion)
    intencion, ent = resultado["intencion"], resultado["entidades"]
    norm = normalizar_texto(texto)

    if intencion != "fallback":
        sesion["fallbacks_consecutivos"] = 0

    if intencion == "entrada_vacia":
        respuesta = RESPUESTAS["entrada_vacia"]
    elif intencion == "emergencia":
        sesion["pendiente"] = None            # la emergencia interrumpe cualquier flujo activo
        respuesta = RESPUESTAS["emergencia"]
    elif intencion == "encuadre_clinico":
        sesion["pendiente"] = None
        respuesta = RESPUESTAS["encuadre_clinico"]
    elif intencion == "contener_frustracion":
        respuesta = responder_contencion(sesion)
    elif intencion == "saludo":
        respuesta = RESPUESTAS["saludo"]
    elif intencion == "consultar_informacion":
        respuesta = responder_informacion(ent)
    elif intencion == "orientacion_homeopatia":
        respuesta = RESPUESTAS["orientacion_homeopatia"]
    elif intencion == "agendar_cita":
        respuesta = gestionar_agendamiento(ent, sesion)
    elif intencion == "consultar_cita":
        respuesta = gestionar_consulta_cita(ent, sesion)
    elif intencion == "derivar_humano":
        sesion["pendiente"] = None
        clave = "derivar_fuera_alcance" if re.search(PATRON_FUERA_ALCANCE, norm) else "derivar_humano"
        respuesta = RESPUESTAS[clave].format(contacto=DATOS_CONTACTO)
    elif intencion == "despedida":
        respuesta = RESPUESTAS["despedida"]
        # Se limpia el estado temporal; las citas de prueba se conservan durante la ejecución
        sesion.update({"pendiente": None, "contador_frustracion": 0, "fallbacks_consecutivos": 0})
    else:  # fallback
        sesion["fallbacks_consecutivos"] += 1
        clave = "fallback_derivacion" if sesion["fallbacks_consecutivos"] >= 2 else "fallback"
        respuesta = RESPUESTAS[clave].format(contacto=DATOS_CONTACTO)

    # Registro estructurado del turno
    sesion["historial"].append({
        "turno": len(sesion["historial"]) + 1,
        "mensaje": texto,
        "intencion": intencion,
        "regla": resultado["regla"],
        "entidades": ent,
        "respuesta": respuesta,
    })
    return {"intencion": intencion, "regla": resultado["regla"], "entidades": ent, "respuesta": respuesta}

sesion_demo = crear_sesion()
for m in ["Hola", "¿Cuánto cuesta la consulta de homeopatía?", "Quiero una cita de homeopatía el jueves a las 11"]:
    r = responder(m, sesion_demo)
    print(f"Usuario: {m}\n  [{r['intencion']} | {r['regla']} | {r['entidades']}]\nInti: {r['respuesta']}\n")

Usuario: Hola
  [saludo | raices | {}]
Inti: Hola, soy Inti, asistente automatizada del Consultorio Integra Salud. Puedo darle información del consultorio (horarios, ubicación, contacto, servicios, costos, formas de pago y documentos), orientarle sobre la consulta de homeopatía, registrar una cita de prueba, consultar una cita registrada o comunicarle con una persona. No puedo dar orientación médica. Esta es una versión de prueba con datos ficticios: por favor no comparta datos personales ni de salud. ¿En qué le puedo ayudar?

Usuario: ¿Cuánto cuesta la consulta de homeopatía?
  [consultar_informacion | raices | {'tema': 'costos', 'servicio': 'homeopatia_primera', 'tipo_asumido': True}]
Inti: Estos son los costos de prueba: Consulta de homeopatía, primera vez: $1,000 MXN (60 minutos). Consulta de homeopatía, seguimiento: $700 MXN (45 minutos). ¿Desea agendar una cita?

Usuario: Quiero una cita de homeopatía el jueves a las 11
  [agendar_cita | entidades_completas | {'servicio': 'homeop

## 9. Interfaz de interacción en el cuaderno

Ejecute la siguiente celda para conversar con Inti. Escriba **salir** para terminar; el historial se devuelve como tabla.

In [13]:
def iniciar_chat():
    sesion = crear_sesion()
    print("Inti:", RESPUESTAS["saludo"])
    print("(Escriba 'salir' para terminar)\n")
    while True:
        try:
            mensaje = input("Usted: ")
        except (EOFError, KeyboardInterrupt):
            break
        if normalizar_texto(mensaje) in ("salir", "exit"):
            print("Inti: Hasta pronto.")
            break
        print("Inti:", responder(mensaje, sesion)["respuesta"], "\n")
    return pd.DataFrame(sesion["historial"])

In [14]:
# Ejecutar para conversar con el chatbot
historial_chat = iniciar_chat()
historial_chat

Inti: Hola, soy Inti, asistente automatizada del Consultorio Integra Salud. Puedo darle información del consultorio (horarios, ubicación, contacto, servicios, costos, formas de pago y documentos), orientarle sobre la consulta de homeopatía, registrar una cita de prueba, consultar una cita registrada o comunicarle con una persona. No puedo dar orientación médica. Esta es una versión de prueba con datos ficticios: por favor no comparta datos personales ni de salud. ¿En qué le puedo ayudar?
(Escriba 'salir' para terminar)

Usted: Hola, buenos días
Inti: Hola, soy Inti, asistente automatizada del Consultorio Integra Salud. Puedo darle información del consultorio (horarios, ubicación, contacto, servicios, costos, formas de pago y documentos), orientarle sobre la consulta de homeopatía, registrar una cita de prueba, consultar una cita registrada o comunicarle con una persona. No puedo dar orientación médica. Esta es una versión de prueba con datos ficticios: por favor no comparta datos perso

,turno,mensaje,intencion,regla,entidades,respuesta
0,1,"Hola, buenos días",saludo,raices,{},"Hola, soy Inti, asistente automatizada del Con..."
1,2,¿A qué hora abren el sábado?,consultar_informacion,raices,"{'tema': 'horarios', 'dia': 'sabado'}",Nuestro horario es el siguiente. Lunes a viern...
2,3,¿Cuánto cuesta la consulta de homeopatía?,consultar_informacion,raices,"{'tema': 'costos', 'servicio': 'homeopatia_pri...",Estos son los costos de prueba: Consulta de ho...
3,4,Qué es la homeopatía?,orientacion_homeopatia,raices,"{'servicio': 'homeopatia_primera', 'tipo_asumi...",La consulta de homeopatía es una valoración qu...
4,5,Quiero agendar una cita,agendar_cita,raices,{},Para registrar su cita de prueba necesito el s...
5,6,Quiero una cita de medicina general el jueves ...,agendar_cita,contexto_pendiente,"{'servicio': 'medicina_general_primera', 'tipo...",Su cita de prueba quedó registrada. Servicio: ...
6,7,Cita de homeopatía de seguimiento el lunes a l...,agendar_cita,entidades_completas,"{'servicio': 'homeopatia_subsecuente', 'dia': ...",El lunes a las 10:00 no hay espacio disponible...
7,8,IS999,consultar_cita,contexto_id_cita,{'id_cita': 'IS999'},No encontré una cita con el identificador IS99...
8,9,Necesito cancelar mi cita,derivar_humano,raices,{},La cancelación y el reagendamiento de citas to...
9,10,Tengo un dolor muy fuerte en el pecho,emergencia,filtro_emergencia,{},Lo que describe puede requerir atención inmedi...


## 10. Pruebas funcionales automatizadas

### 10.1 Matriz de pruebas por intención

Cada caso se ejecuta en una sesión nueva, salvo los que dependen de una cita previa. Una prueba se aprueba cuando la intención detectada coincide con la esperada **y** la respuesta contiene el fragmento esperado.

In [15]:
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

CASOS = [
    # (id, intención esperada, entrada, fragmento esperado en la respuesta, tipo de prueba)
    ("P01", "saludo", "Hola, buenos días", "soy Inti", "representativa"),
    ("P02", "saludo", "HOLA INTI!!!", "soy Inti", "mayúsculas y signos"),
    ("P03", "consultar_informacion", "¿A qué hora abren el sábado?", "Sábado: 10:00 a 14:00", "representativa"),
    ("P04", "consultar_informacion", "cuando atienden", "Lunes a viernes", "sin signos ni acentos"),
    ("P05", "consultar_informacion", "¿Dónde están ubicados?", "Av. Ejemplo 123", "representativa"),
    ("P06", "consultar_informacion", "¿Cuál es su teléfono?", "55 0000 0000", "representativa"),
    ("P07", "consultar_informacion", "¿Qué servicios ofrecen?", "certificado médico", "representativa"),
    ("P08", "consultar_informacion", "¿Cuánto cuesta la consulta de homeopatía?", "$1,000 MXN", "ambigua (costos y homeopatía)"),
    ("P09", "consultar_informacion", "¿Aceptan tarjeta de crédito?", "transferencia", "representativa"),
    ("P10", "consultar_informacion", "¿Qué documentos debo llevar?", "identificación oficial", "representativa"),
    ("P11", "orientacion_homeopatia", "¿Qué es la homeopatía?", "complementaria", "representativa"),
    ("P12", "orientacion_homeopatia", "en que consiste la primera consulta homeopatica", "sesenta minutos", "variación"),
    ("P13", "agendar_cita", "Quiero agendar una cita", "en un solo mensaje", "datos incompletos"),
    ("P14", "agendar_cita", "Quiero una cita de medicina general el jueves a las 11", "IS101", "solicitud completa"),
    ("P15", "agendar_cita", "Agéndame homeopatía el domingo a las 12", "no atiende en domingo", "regla de negocio"),
    ("P16", "agendar_cita", "Cita de homeopatía de seguimiento el lunes a las 10", "no hay espacio", "bloque ocupado"),
    ("P17", "consultar_cita", "¿Cuándo es mi cita?", "identificador", "sin identificador"),
    ("P18", "consultar_cita", "IS999", "No encontré", "identificador inexistente"),
    ("P19", "derivar_humano", "Quiero hablar con una persona", "atención personal", "representativa"),
    ("P20", "derivar_humano", "Necesito cancelar mi cita", "no están disponibles", "fuera de alcance del MVP"),
    ("P21", "emergencia", "Tengo un dolor muy fuerte en el pecho", "911", "seguridad"),
    ("P22", "emergencia", "no puedo respirar bien", "911", "seguridad"),
    ("P23", "emergencia", "Quiero hacerme daño", "800 911 2000", "seguridad"),
    ("P24", "encuadre_clinico", "Tengo fiebre, ¿qué medicamento me tomo?", "no puedo valorar", "límite clínico"),
    ("P25", "contener_frustracion", "Estoy harto, pésimo servicio", "lamento la molestia", "frustración"),
    ("P26", "contener_frustracion", "eres un inútil", "lamento la molestia", "lenguaje agresivo"),
    ("P27", "despedida", "Muchas gracias, hasta luego", "Que tenga un buen día", "representativa"),
    ("P28", "despedida", "eso es todo", "Que tenga un buen día", "variación"),
    ("P29", "fallback", "¿Cómo estará el clima mañana?", "No estoy segura", "fuera de dominio"),
    ("P30", "fallback", "Recomiéndame una película", "No estoy segura", "fuera de dominio"),
    ("P31", "entrada_vacia", "", "No recibí ningún texto", "entrada vacía"),
    ("P32", "entrada_vacia", "???", "No recibí ningún texto", "solo signos"),
    ("P33", "consultar_informacion", "Hola, ¿cuánto cuesta el certificado médico?", "$500 MXN", "ambigua (saludo y costos)"),
    ("P34", "consultar_informacion", "q horario tienen", "Lunes a viernes", "abreviatura"),
    ("P35", "consultar_informacion", "dirrecion del consultorio", "Av. Ejemplo 123", "error ortográfico"),
    ("P36", "consultar_cita", "IS101", "Encontré su cita", "consulta de cita existente"),
]

def ejecutar_matriz(casos):
    filas = []
    sesion_compartida = crear_sesion()
    for id_prueba, esperada, entrada, fragmento, tipo in casos:
        # P14 y P36 comparten sesión para que la cita registrada pueda consultarse
        sesion = sesion_compartida if id_prueba in ("P14", "P36") else crear_sesion()
        r = responder(entrada, sesion)
        aprobada = r["intencion"] == esperada and fragmento in r["respuesta"]
        filas.append({
            "id": id_prueba,
            "tipo": tipo,
            "entrada": entrada,
            "intencion_esperada": esperada,
            "fragmento_esperado": fragmento,
            "intencion_observada": r["intencion"],
            "regla": r["regla"],
            "respuesta_observada": r["respuesta"],
            "estado": "Aprobada" if aprobada else "No aprobada",
        })
    return pd.DataFrame(filas)

df_matriz = ejecutar_matriz(CASOS)
df_matriz[["id", "tipo", "entrada", "intencion_esperada", "intencion_observada", "regla", "estado"]]

,id,tipo,entrada,intencion_esperada,intencion_observada,regla,estado
0,P01,representativa,"Hola, buenos días",saludo,saludo,raices,Aprobada
1,P02,mayúsculas y signos,HOLA INTI!!!,saludo,saludo,raices,Aprobada
2,P03,representativa,¿A qué hora abren el sábado?,consultar_informacion,consultar_informacion,raices,Aprobada
3,P04,sin signos ni acentos,cuando atienden,consultar_informacion,consultar_informacion,raices,Aprobada
4,P05,representativa,¿Dónde están ubicados?,consultar_informacion,consultar_informacion,raices,Aprobada
5,P06,representativa,¿Cuál es su teléfono?,consultar_informacion,consultar_informacion,raices,Aprobada
6,P07,representativa,¿Qué servicios ofrecen?,consultar_informacion,consultar_informacion,raices,Aprobada
7,P08,ambigua (costos y homeopatía),¿Cuánto cuesta la consulta de homeopatía?,consultar_informacion,consultar_informacion,raices,Aprobada
8,P09,representativa,¿Aceptan tarjeta de crédito?,consultar_informacion,consultar_informacion,raices,Aprobada
9,P10,representativa,¿Qué documentos debo llevar?,consultar_informacion,consultar_informacion,raices,Aprobada


In [16]:
# Métricas vistas en clase: precisión de intención y tasa de fallback
precision = (df_matriz["intencion_esperada"] == df_matriz["intencion_observada"]).mean()
aprobacion = (df_matriz["estado"] == "Aprobada").mean()
tasa_fallback = (df_matriz["intencion_observada"] == "fallback").mean()

print(f"Casos ejecutados:          {len(df_matriz)}")
print(f"Precisión de intención:    {precision:.1%}")
print(f"Pruebas aprobadas:         {aprobacion:.1%}")
print(f"Tasa de fallback:          {tasa_fallback:.1%}")

seguridad = df_matriz[df_matriz["intencion_esperada"].isin(["emergencia", "encuadre_clinico", "contener_frustracion"])]
print(f"Detección de filtros de seguridad: {(seguridad['estado'] == 'Aprobada').mean():.0%} ({len(seguridad)} casos)")

print("\nPruebas no aprobadas:")
df_matriz[df_matriz["estado"] == "No aprobada"][["id", "entrada", "intencion_esperada", "intencion_observada", "regla"]]

Casos ejecutados:          36
Precisión de intención:    97.2%
Pruebas aprobadas:         97.2%
Tasa de fallback:          8.3%
Detección de filtros de seguridad: 100% (6 casos)

Pruebas no aprobadas:


,id,entrada,intencion_esperada,intencion_observada,regla
34,P35,dirrecion del consultorio,consultar_informacion,fallback,umbral_no_alcanzado


In [17]:
# Análisis de los casos no aprobados: qué raíces quedaron después del preprocesamiento
for _, fila in df_matriz[df_matriz["estado"] == "No aprobada"].iterrows():
    pre = preprocesar(fila["entrada"])
    _, puntajes = clasificar_por_raices(pre["raices"])
    print(f"{fila['id']} | {fila['entrada']}")
    print(f"   raíces: {pre['raices']}")
    print(f"   puntajes: { {k: v for k, v in puntajes.items() if v} or 'ninguna coincidencia'}\n")

P35 | dirrecion del consultorio
   raíces: ['dirrecion']
   puntajes: ninguna coincidencia



### 10.2 Conversaciones de prueba

Se ejecutan trece conversaciones de varios turnos, cada una en una sesión nueva, para comprobar la coherencia del diálogo: agendamiento en dos turnos, consulta de la cita creada, reglas de negocio, interrupción por emergencia, escalamiento de la contención y derivación tras fallbacks repetidos.

In [18]:
CONVERSACIONES = {
    "C01 Paciente nuevo se informa": [
        "Hola", "¿Qué servicios ofrecen?", "¿Cuánto cuesta la homeopatía?", "¿Aceptan tarjeta?", "Gracias"],
    "C02 Agendamiento en dos turnos": [
        "Buenas tardes", "Quiero agendar una cita", "Homeopatía primera vez el martes a las 4 pm", "Gracias, hasta luego"],
    "C03 Agendar y consultar la cita": [
        "Quiero una cita de medicina general de seguimiento el viernes a las 12", "¿Cuándo es mi cita?", "IS101"],
    "C04 Bloque ocupado y reintento": [
        "Cita de homeopatía el lunes a las 10", "Homeopatía el lunes a las 11"],
    "C05 Día cerrado y reintento": [
        "Agéndame un certificado médico el domingo a las 11", "Certificado médico el sábado a las 11"],
    "C06 Orientación antes de agendar": [
        "¿Qué es la homeopatía?", "¿En qué se diferencia de la alopatía?", "Quiero agendar homeopatía el miércoles a las 13:00"],
    "C07 Emergencia interrumpe el flujo": [
        "Quiero agendar una cita", "Me duele mucho el pecho y no puedo respirar", "Gracias"],
    "C08 Contención con escalamiento": [
        "No entiendes nada", "Estoy harto, pésimo servicio", "Eres un inútil"],
    "C09 Fallback repetido y derivación": [
        "¿Cómo estará el clima mañana?", "Recomiéndame una película", "Quiero hablar con una persona"],
    "C10 Solicitud fuera de alcance": [
        "Necesito cancelar mi cita", "¿Y puedo reagendarla?"],
    "C11 Encuadre clínico y agendamiento": [
        "Tengo tos y fiebre desde ayer", "¿Qué medicamento me tomo?", "Quiero medicina general el jueves a las 17"],
    "C12 Entradas vacías o inesperadas": [
        "", "   ", "😀", "hola"],
    "C13 Documentos, ubicación y contacto": [
        "¿Qué documentos debo llevar?", "¿Dónde están ubicados?", "¿Cuál es su teléfono?", "Adiós"],
}

registros = []
for nombre, mensajes in CONVERSACIONES.items():
    sesion = crear_sesion()
    print("=" * 100)
    print(nombre)
    print("=" * 100)
    for mensaje in mensajes:
        r = responder(mensaje, sesion)
        print(f"Usuario: {mensaje!r}")
        print(f"   [{r['intencion']} | {r['regla']}]")
        print(f"Inti: {r['respuesta']}\n")
        registros.append({"conversacion": nombre, "mensaje": mensaje, "intencion": r["intencion"],
                          "regla": r["regla"], "respuesta": r["respuesta"]})

df_conversaciones = pd.DataFrame(registros)
print(f"Conversaciones: {df_conversaciones['conversacion'].nunique()} | Turnos: {len(df_conversaciones)}")

C01 Paciente nuevo se informa
Usuario: 'Hola'
   [saludo | raices]
Inti: Hola, soy Inti, asistente automatizada del Consultorio Integra Salud. Puedo darle información del consultorio (horarios, ubicación, contacto, servicios, costos, formas de pago y documentos), orientarle sobre la consulta de homeopatía, registrar una cita de prueba, consultar una cita registrada o comunicarle con una persona. No puedo dar orientación médica. Esta es una versión de prueba con datos ficticios: por favor no comparta datos personales ni de salud. ¿En qué le puedo ayudar?

Usuario: '¿Qué servicios ofrecen?'
   [consultar_informacion | raices]
Inti: Ofrecemos consulta de medicina general, consulta de homeopatía (primera vez y seguimiento) y expedición de certificado médico. Si quiere, le comparto los costos o le explico en qué consiste la consulta de homeopatía.

Usuario: '¿Cuánto cuesta la homeopatía?'
   [consultar_informacion | raices]
Inti: Estos son los costos de prueba: Consulta de homeopatía, prime

In [19]:
# Exportación de evidencias para el repositorio de GitHub
import os
os.makedirs("pruebas", exist_ok=True)
df_matriz.to_csv("pruebas/matriz_pruebas.csv", index=False, encoding="utf-8-sig")
df_conversaciones.to_csv("pruebas/conversaciones.csv", index=False, encoding="utf-8-sig")
print("Archivos guardados en la carpeta pruebas/")

Archivos guardados en la carpeta pruebas/


## 11. Análisis de resultados y limitaciones

Los valores de esta sección se obtienen al ejecutar las celdas anteriores. Si se modifica el catálogo, vuelva a ejecutar el cuaderno completo y actualice el análisis.

**Funcionamiento observado.** Las siete intenciones conversacionales, los dos filtros de seguridad, la regla de encuadre clínico y el fallback responden según lo esperado en los casos representativos. Las reglas de contexto permiten completar el agendamiento en dos turnos y consultar la cita creada con su identificador.

**Limitaciones observadas en las pruebas**

1. **Errores ortográficos fuera del vocabulario.** "dirrecion" no comparte raíz con "direccion", por lo que el mensaje cae en fallback (prueba P35).
2. **Dependencia de palabras clave.** Una intención solo se reconoce si el mensaje contiene alguna raíz presente en sus ejemplos; un sinónimo no registrado no se detecta.

**Limitaciones previsibles por el diseño**

1. El agendamiento exige servicio, día y hora en un mismo mensaje y solo reconoce días de la semana, no fechas ni expresiones como "mañana".
2. Los filtros de seguridad por palabras clave pueden producir falsos positivos (por ejemplo, "¿atienden urgencias?" activa el mensaje de emergencia) y no detectan expresiones fuera de su lista.
3. El estado de la sesión vive en memoria y se pierde al reiniciar el entorno.

**Mejoras propuestas**

| Limitación | Mejora | Momento |
|---|---|---|
| Errores ortográficos | Comparación aproximada de cadenas (distancia de edición con `nltk.edit_distance`) contra el vocabulario | Inmediata |
| Sinónimos no registrados | Ampliar ejemplos con frases reales y, después, un clasificador supervisado o *embeddings* | Etapa de PLN |
| Agendamiento en un mensaje | Llenado de *slots* en varios turnos y reconocimiento de fechas relativas | Siguiente versión |
| Estado en memoria | Persistencia ligera (SQLite) e integración con un calendario real | Integración web |